A notebook to walk the latest space.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from normalizing_flows.src.utils import load_config
from normalizing_flows.src.glow.model.glow_flow import Glow
from normalizing_flows.src.realnvp.callbacks import ModelCheckpoint
from normalizing_flows.scripts.train_celeba import create_celeba_dataset, create_model


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Init dataset and load trained model

In [ ]:
config = load_config('../experiments/glow128x128_6/config_glow128x128_6.yaml')

In [ ]:
ds, _ = create_celeba_dataset(split='test', n_chunks=1, ds_config=config['dataset'])

In [ ]:
model = create_model(config['model'])
model = model.eval()
ModelCheckpoint.load(model, checkpoint_path='../experiments/glow128x128_6/checkpoints/realnvp_075_1.870.pt')

# Load random images and plot

In [ ]:
# Select random images
images = []
for i in np.random.randint(0, len(ds), size=4):
    img, _ = ds[i]
    images.append(img)

In [ ]:
def walk_latent_space(model, img0, img1, img2, img3, num=7):
    z0, _ = model(img0.unsqueeze(0).to(device))
    z1, _ = model(img1.unsqueeze(0).to(device))
    z2, _ = model(img2.unsqueeze(0).to(device))
    z3, _ = model(img3.unsqueeze(0).to(device))
    
    fig, axes = plt.subplots(num, num, figsize=(3*num, 3*num))

    # The inference could be vectorized for speed, but it I don't really need speed right now
    for i, w_hor in enumerate(np.linspace(0, 1, num=num)):
        for j, w_vert in enumerate(np.linspace(0, 1, num=num)):
            # Calculate weights for each latent tensor
            w0 = (1 - w_hor) * (1 - w_vert)  # top left
            w1 = w_hor * (1 - w_vert)  # top right
            w2 = (1 - w_hor) * w_vert  # bottom left
            w3 = w_hor * w_vert  # bottom right
            # Linearly interpolate between the four tensors
            z_inter = z0*w0 + z1*w1 + z2*w2 + z3*w3
            # Inverse from latent space to image space
            img_inter = model.inverse(z_inter)
            # Convert to numpy as plot image
            img_inter = img_inter.cpu().numpy()[0].transpose(1, 2, 0)
            ax = axes[i, j]
            ax.imshow(img_inter)
            ax.axis('off')
    fig.tight_layout(pad=0.25)

In [ ]:
walk_latent_space(model, *images, num=7)